# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes the accepted jobs to `jobs/2-classified-jobs.jsonl`.


In [7]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

### Load scraped jobs from file

In [8]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

if jobs_df.empty:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first."
    )

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 14 entries!


### Define the prompt

We define the instructions that tell the model what counts as an AI engineering role.

In [9]:
client = OpenAI()

instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- An AI Engineer is not mainly a model researcher and does not primarily build models from scratch.
- AI engineering is closer to product and application engineering than to core ML research.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

### Define the schema

We tell the model to return structured output with a boolean decision and a short reason.

In [10]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Make the LLM calls

We now ask the model to classify each job posting.

In [ ]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = result["is_ai_engineering_role"]
    reason = result["reason"].strip()

    results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

Screening job 1/14: Senior ML/AI Engineer
Screening job 2/14: AI (Artificial Intelligence) Engineer (Remote)
Screening job 3/14: AI Engineer
Screening job 4/14: Lead AI Engineer (AI Foundations)
Screening job 5/14: Oracle AI Engineer - Senior Manager
Screening job 6/14: AI Engineer
Screening job 7/14: AI Engineer
Screening job 8/14: Python AI Engineer
Screening job 9/14: AI Engineer
Screening job 10/14: AI Engineer
Screening job 11/14: AI Engineer Intern
Screening job 12/14: AI/Generative AI Engineer
Screening job 13/14: AI Engineer
Screening job 14/14: Senior ML/AI Engineer


### Combine the results

We join the model output back to the scraped jobs and save the accepted jobs

In [12]:
results_df = pd.DataFrame(results)
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

ai_engineer_jobs = screened_jobs[screened_jobs["is_ai_engineering_role"]].copy()
non_ai_engineer_count = int((~screened_jobs["is_ai_engineering_role"]).sum())
ai_engineer_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

### Display the results

In [13]:
print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {len(ai_engineer_jobs)}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

Saved classified jobs to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/jobs/2-classified-jobs.jsonl
Jobs screened by LLM: 14
Jobs classified as AI engineering roles: 11
Jobs classified as not AI engineering or unclear: 3


title,is_ai_engineering_role,llm_reason,job_url
Senior ML/AI Engineer,True,"The role is centered on building a user-facing conversational interface on top of existing ML/LLM systems, with responsibilities like deploying models, optimizing inference, and shipping AI-powered product features. It is application/product-focused rather than model research or training from scratch.",https://www.indeed.com/viewjob?jk=c146335910a915d7
AI (Artificial Intelligence) Engineer (Remote),True,"The role centers on building application features on top of LLMs/AI agents: designing, integrating, and implementing LLM-based solutions, prompt engineering, orchestration, RAG/agent workflows, and API/backend integration. It is not primarily model training, research, or classical ML engineering.",https://www.indeed.com/viewjob?jk=b8b25b4aaf482123
AI Engineer,True,"The role is centered on building application-layer AI systems on top of foundation models: agent architectures, prompt/retrieval/context engineering, tool calling, evaluation, feedback loops, and production deployment. It is not primarily model training, research, or classical ML/platform work.",https://www.indeed.com/viewjob?jk=c3d5ae2f3753dbff
Lead AI Engineer (AI Foundations),True,"The role centers on building and deploying AI-powered products and software components on top of foundation models/LLMs (inference, vector search, guardrails, evaluation, observability, optimization). It is application-focused AI engineering rather than model research or traditional ML training from scratch.",https://www.indeed.com/viewjob?jk=85bdd2fdebefde9a
Oracle AI Engineer - Senior Manager,True,"The posting centers on designing and delivering AI and generative AI solutions, especially GenAI application development, over existing models. It emphasizes leading AI workstreams, productionizing proofs of concept, and integrating AI into enterprise systems, which fits AI engineering rather than model research or classical ML.",https://www.indeed.com/viewjob?jk=479942b34465086e
AI Engineer,False,"The posting includes AI work, but the role is not primarily AI engineering as defined. It emphasizes end-to-end ML model development/training/fine-tuning, MLOps, data engineering, and enterprise software architecture rather than building product features mainly on top of existing foundation models/LLMs. It is closer to traditional ML engineering plus software/platform work.",https://www.linkedin.com/jobs/view/4395309456
AI Engineer,True,"The role is explicitly focused on building AI applications on top of LLMs/foundation models: agentic systems, orchestration, prompt/context engineering, production APIs, and AI evaluation/observability. It is application engineering rather than model training, research, or classical ML/MLOps.",https://www.linkedin.com/jobs/view/4393576713
Python AI Engineer,True,"The role centers on building production AI agents and workflows on top of foundation models using LangGraph, RAG, prompt engineering, and text-to-SQL. This is application-level AI engineering rather than model training, research, or traditional ML/platform work.",https://www.linkedin.com/jobs/view/4392455560
AI Engineer,False,"The role is primarily operational support, AI platform administration, CI/CD, access control, compliance, and Azure/MLOps-style maintenance. It does not center on building product features on top of foundation models/LLMs, so it is not a true AI engineering role by the given definition.",https://www.linkedin.com/jobs/view/4395041804
AI Engineer,False,"The posting is heavily focused on building, training, deploying, and optimizing AI/ML models, MLOps, and infrastructure rather than primarily building application features on top of existing foundation models. It includes LLM/RAG work, but that is not the core emphasis.",https://www.linkedin.com/jobs/view/4393208617
